In [16]:
import asyncio
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [17]:
def print_last_message(result):
    messages = result.get("messages", [])
    if not messages:
        print(result)
        return
    msg = messages[-1]
    content = msg.get("content") if isinstance(msg, dict) else getattr(msg, "content", None)
    print(content if content is not None else msg)

In [18]:
llm = ChatOpenAI(
openai_api_base= "http://10.0.1.1:10000/v1/",
openai_api_key= "",
model_name="nvidia/Nemotron-Orchestrator-8B"
)

In [19]:
mcp_client = MultiServerMCPClient(
    {
        "cpq-models-health-check": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    }
)

In [20]:
TOOLS = await mcp_client.get_tools(server_name="cpq-models-health-check")

print("ДОСТУПНЫЕ ИНСТРУМЕНТЫ")
print("*"* 100)
for x in TOOLS:
    print(f"name: {x.name} secs: {x.description}")
print("*"* 100)

ДОСТУПНЫЕ ИНСТРУМЕНТЫ
****************************************************************************************************
name: check_binary_type_compatibility secs: Проверка на соответствие бинарников и типов моделей
name: check_predictor_count_compatibility secs: Проверка соответствия количества предикторов в бинарнике и конфигурации
name: check_predictors_without_recent_values secs: Проверка предикторов, у которых нет значений за последний час
name: find_disabled_models secs: Поиск отключенных моделей через поле disabled
name: find_models_with_different_units secs: Поиск моделей с разными единицами измерения с их ЛА
name: find_models_with_frozen_values secs: Поиск моделей с неизменяющимися значениями за последний час
name: find_models_with_limited_values secs: Поиск моделей, у которых последнее значение упирается в пределы tech_min или tech_max
name: find_models_without_evaluation_date secs: Поиск моделей без даты добавления модели
name: find_models_without_ipk_config secs: Поиск м

In [21]:
SYSTEM_PROMPT = """
Отвечай только на русском языке.
Ты — эксперт-аудитор ML-моделей в системе контроль и прогноз какчества (КПК).
Большая часть моделей - виртуальные анализаторы качества либо модели смешения.
Твоя задача — выявлять проблемы моделей по запросам пользователя: отсутствие статусов/тегов/значений/дат, отключённые модели, несоответствия конфигов (IPK/train), пределов (tech_min/max), предикторов, бинарников и данных (за час/последние).


Правила:
1. Для запроса о проблемах — вызови **все релевантные** инструменты (цепочкой, если нужно).
2. Собери результаты, проанализируй: сгруппируй по типам проблем, посчитай модели, выдели критику (отключённые, без данных).
3. Ответ: Markdown-таблица с колонками "Проблема | Инструмент | Кол-во моделей | Примеры моделей".
4. Если нет проблем — скажи "OK".
5. Только факты из инструментов, без домыслов.
6. Финал: "Рекомендации: [кратко, напр. обновить статусы]".
"""

In [25]:
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

In [28]:
MESSAGE = "Выведи все модели, которые на текущий момент упираются в максимальные или минимальные пределы."

result = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": MESSAGE,
            }
        ]
    }
)

print_last_message(result)

<think>
Okay, let me process this. The user asked for models that are hitting their max or min limits. The tool response shows 10 models. Each has tech_min, tech_max, and last_value. I need to check if last_value equals tech_min or tech_max.

Looking at the first entry: last_value is 0.9, tech_max is 0.9, so max_bound is true. Then the next one has last_value 0.0, tech_min 0.0, so min_bound is true. I need to categorize each model as either hitting the min, max, or both. 

Wait, some models might have both min and max bounds. For example, if a model's last_value is exactly tech_min and tech_max, but that's unlikely unless tech_min equals tech_max. But in the data, tech_min and tech_max are different for each model. So each model is either at min, max, or neither. 

I need to group them into two categories: those hitting the minimum and those hitting the maximum. Then count how many models fall into each category. Also, check if any model is hitting both, but looking at the data, I don'